# DataPulse AI — Google Cloud Free Trial + Vertex AI

**Use the correct billing route:** GCP trial credits (roughly ₹28,000 if your account displays that amount) are not the same as AI Studio/Gemini API prepayment credits. An alerts-only ₹1,000 budget is just a notification threshold, not a payment, limit, or deduction. **Never click Upgrade/Activate billing automatically.** Trial eligibility and model access vary.

This notebook uses your Google Cloud project through Vertex AI, and **falls back to a truthful deterministic incident explanation if model access is not available**. It never needs an AI Studio `GOOGLE_API_KEY`.


## 1 — Fetch the project and install SDK


In [ ]:
import os, subprocess
from pathlib import Path
repo = Path('/content/aegis-real-time-supply-chain-ai')
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git',str(repo)],check=True)
os.chdir(repo / 'datapulse-ai')
print('Project directory:',os.getcwd())
%pip -q install -r requirements.txt google-genai


from google.colab import auth
from google import genai

PROJECT_ID = 'REPLACE_WITH_YOUR_GCP_PROJECT_ID'  # use PROJECT ID (not project name)
LOCATION = 'global'
assert PROJECT_ID != 'REPLACE_WITH_YOUR_GCP_PROJECT_ID', 'Replace the project ID first'

# Colab authentication uses your Google identity, not an AI Studio key.
auth.authenticate_user()

vertex_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
print('Vertex AI configured for:', PROJECT_ID, 'region:', LOCATION)


In [ ]:
from google.colab import auth
from google import genai

auth.authenticate_user()
PROJECT_ID = 'REPLACE_WITH_YOUR_GCP_PROJECT_ID'  # Project ID, not display name
LOCATION = 'global'  # Use global endpoint for broad model availability
assert PROJECT_ID != 'REPLACE_WITH_YOUR_GCP_PROJECT_ID', 'Set your actual project ID first'
vertex_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
print('Vertex AI client configured for project:', PROJECT_ID)


MODEL_NAME = 'gemini-3.8-flash'
VERTEX_OK = False
try:
    connection_test = vertex_client.models.generate_content(
        model=MODEL_NAME,
        contents='Reply with only: DataPulse Vertex AI connected.'
    )
    VERTEX_OK = True
    print('VERTEX AI CONNECTED:', connection_test.text)
except Exception as exc:
    print('VERTEX AI UNAVAILABLE (local DataPulse still works):', type(exc).__name__)
    print(str(exc)[:1800])
    print('\n403=IAM/API/trial; 404=model availability; 429=quota; 402=billing. No need to rerun ML or buy credits.')


In [ ]:
MODEL_NAME = 'gemini-3.8-flash'
try:
    connection_test = vertex_client.models.generate_content(
        model=MODEL_NAME,
        contents='Reply with only: DataPulse Vertex AI connected.'
    )
    print('VERTEX AI RESPONSE:', connection_test.text)
except Exception as exc:
    print('VERTEX AI ERROR:', type(exc).__name__)
    print(str(exc)[:1800])
    raise


## 4 — Rebuild the verified DataPulse incident report

The ML results and root-cause report are constructed locally. The following call makes **no** Vertex AI request.


In [ ]:
import json
from datapulse.replay import decision_replay
from datapulse.investigator import investigate
summary, comparisons = decision_replay()
report = investigate(summary)
assert isinstance(report,dict) and 'evidence' in report and 'incident_type' in report
print(json.dumps(report,indent=2))


if VERTEX_OK:
    try:
        response = vertex_client.models.generate_content(model=MODEL_NAME, contents=prompt)
        print('VERTEX AI GEMINI INCIDENT EXPLANATION\n')
        print(response.text)
    except Exception as exc:
        print('Explanation call failed:', type(exc).__name__, str(exc)[:1200])
        VERTEX_OK = False
if not VERTEX_OK:
    print('LOCAL EVIDENCE-BASED INCIDENT EXPLANATION (no Gemini API used)')
    print('Incident:', report['incident_type'])
    print('Evidence:', json.dumps(report['evidence'], indent=2))
    print('Recommendation:', report['recommended_action'])
    print('Approval required:', report['requires_human_approval'])
    print('Limitations:', report['limitations'])
    print('\nNext: verify BigQuery tables and Pub/Sub event. Do not claim a successful Gemini call.')


In [ ]:
prompt = ('You are a read-only DataPulse AI incident assistant. Explain the verified JSON incident evidence, '
          'why basic validation missed the semantic shift, model decision impact, next checks, and human-approved actions. '
          'Do not invent figures, claim proven causal business harm, or treat synthetic outcomes as real customers.\n\n'
          + json.dumps(report, sort_keys=True))
response = vertex_client.models.generate_content(model=MODEL_NAME, contents=prompt)
print('DATAPULSE AI / VERTEX AI EXPLANATION\n')
print(response.text)


## Troubleshooting

- `403`: Vertex AI API disabled, IAM access, or trial restrictions
- `404`: model name/region or project model availability; try another supported model using [official model documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models)
- `429`: quota or rate limit
- Billing messages: review the **Google Cloud project** billing, not AI Studio prepaid balance
- `GOOGLE_API_KEY` must **not** be needed here

The developer Gemini API and Vertex AI have different authentication and billing paths. Never paste access tokens or API keys into chat or GitHub.


## 6 — Optional: Prove the BigQuery integration with trial credits

These are **cloud operations** that may consume eligible credits. Enable BigQuery API in the Cloud Console first. Run this only after confirming the project ID and trial status. This produces three synthetic BigQuery tables. No production data are involved.


In [ ]:
%pip -q install -r requirements-gcp.txt
!python -m cloud.bigquery_load --project-id {PROJECT_ID}


## 7 — Optional: Publish a Pub/Sub incident event

Enable Pub/Sub API first. This publishes exactly one synthetic event to `datapulse-incidents` and prints its message ID; it does **not** create a subscription, consumer, or production stream. Don't claim streaming deployment on the basis of this publisher.


In [ ]:
!python -m cloud.pubsub_demo --project-id {PROJECT_ID}


## 8 — Showcase checklist

- Screenshot DataPulse incident JSON and fixed-model replay metrics (local; reproducible)
- Screenshot successful BigQuery table loads and one SQL query (only if actually run)
- Screenshot Pub/Sub published message ID (only if actually run)
- Screenshot Vertex AI Gemini explanation **only if a successful response appears**; otherwise explicitly describe a deterministic offline explanation and trial-access limitation
- Do not publicly share billing information, project identifiers if confidential, or authentication tokens

**GCP console steps:** (1) Select trial project and verify *Billing > Overview* active/free trial and credit balance. (2) Enable APIs: Vertex AI, BigQuery, Pub/Sub. (3) Run notebook cells one at a time. (4) For 403 check the signed-in account has Vertex AI User IAM permission, API enabled, and any trial/model access restriction. (5) If model use is restricted, do not upgrade billing automatically; complete BigQuery and Pub/Sub evidence instead.
